In [1]:
!nvidia-smi

Fri May 29 12:36:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

True
1


In [3]:
# =========================================
# 1. INSTALL LIBRARIES
# =========================================
!pip install torch transformers pandas scikit-learn tqdm imbalanced-learn


In [4]:
# =========================================
# 2. IMPORT LIBRARIES
# =========================================
import os
import pandas as pd
import numpy as np
import torch
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils import resample

from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, get_scheduler

In [5]:
# =========================================
# 3. MOUNT GOOGLE DRIVE
# =========================================
from google.colab import drive
drive.mount('/content/drive')

# Path where model will be saved
MODEL_PATH = "/content/drive/MyDrive/Bert_Senti/spam_model_v2"  # v2 = new retrained model

Mounted at /content/drive


In [6]:

# =========================================
# 4. CHECK DEVICE
# =========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [7]:
# 5. CHECK if model exists
model_exists = os.path.exists(MODEL_PATH)

if model_exists:
    print("Model already trained. Loading from Drive.")
else:
    print("No trained model found. Training will start.")

Model already trained. Loading from Drive.


In [8]:
# 6. Load if model exists
if model_exists:

    model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
    tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)

    model.to(device)
    model.eval()

    print("Model loaded successfully.")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Model loaded successfully.


In [9]:
# =========================================
# 7. LOAD DATASETS IF MODEL NOT TRAINED
# =========================================
if not model_exists:

    # Original SMS datasets
    url1 = 'https://raw.githubusercontent.com/princebari/-SMS-Spam-Classification-on-Indian-Dataset-A-Crowdsourced-Collection-of-Hindi-and-English-Messages/main/indian_spam.csv'
    url2 = 'https://huggingface.co/datasets/thehamkercat/telegram-spam-ham/resolve/main/dataset.csv'
    url3 = 'https://huggingface.co/datasets/SetFit/enron_spam/resolve/main/enron_spam_data.csv'
    url4 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/data-en-hi-de-fr.csv"
    url5 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/indian_sms_spam_dataset_12000.csv"
    url6 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/spam_ham_dataset.csv"
    url7 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/spam.csv"
    url8 = "https://raw.githubusercontent.com/War-rack/Q-Classify/refs/heads/main/messages.csv"

    df1 = pd.read_csv(url1)
    df2 = pd.read_csv(url2)
    df3 = pd.read_csv(url3)
    df4 = pd.read_csv(url4)
    df5 = pd.read_csv(url5)
    df6 = pd.read_csv(url6)
    df7 = pd.read_csv(url7, encoding="latin1")
    df8 = pd.read_csv(url8)

    # New email datasets (upload these to your Drive root first)
    df_emails   = pd.read_csv("/content/drive/MyDrive/Bert_Senti/emails.csv")
    df_phishing = pd.read_csv("/content/drive/MyDrive/Bert_Senti/Phishing_Email.csv")

    print("All datasets loaded!")

In [10]:
# =========================================
# 8. STANDARDIZE COLUMNS
# =========================================
if not model_exists:

    df1 = df1.rename(columns={"v1":"label","v2":"SMS"})[["label","SMS"]]
    df2 = df2.rename(columns={"text_type":"label","text":"SMS"})[["label","SMS"]]
    df3 = df3.rename(columns={"Spam/Ham":"label","Message":"SMS"})[["label","SMS"]]
    df4 = df4.rename(columns={"labels":"label","text":"SMS"})[["label","SMS"]]
    df5 = df5.rename(columns={"text":"SMS"})[["label","SMS"]]
    df6 = df6.drop(columns=["Unnamed: 0","label_num"], errors="ignore")
    df6 = df6.rename(columns={"text":"SMS"})[["label","SMS"]]
    df7 = df7.rename(columns={"v1":"label","v2":"SMS"})[["label","SMS"]]

    # messages.csv
    df8.columns = df8.columns.str.strip().str.lower()
    df8 = df8.rename(columns={"text":"SMS","message":"SMS","messages":"SMS"})[["label","SMS"]] if "label" in df8.columns else df8.rename(columns={df8.columns[0]:"label", df8.columns[1]:"SMS"})[["label","SMS"]]

    # emails.csv → text, spam(0/1)
    df_emails = df_emails.rename(columns={"text":"SMS","spam":"label"})[["label","SMS"]]
    df_emails["label"] = df_emails["label"].astype(int)

    # Phishing_Email.csv → Email Text, Email Type
    df_phishing = df_phishing.rename(columns={"Email Text":"SMS","Email Type":"label"})[["label","SMS"]]
    df_phishing["label"] = df_phishing["label"].replace({
        "Safe Email": 0,
        "Phishing Email": 1
    })

    df = pd.concat([df1,df2,df3,df4,df5,df6,df7,df8,df_emails,df_phishing], ignore_index=True)
    df.drop_duplicates(inplace=True)
    df.dropna(inplace=True)

    print("Dataset size:", df.shape)

In [11]:
# =========================================
# 9. ENCODE LABELS
# =========================================
if not model_exists:

    df["label"] = df["label"].replace({"ham":0,"spam":1}).astype(int)

    print(df["label"].value_counts())

In [12]:
# =========================================
# 10. BALANCE DATASET
# =========================================
if not model_exists:

    df_ham  = df[df.label==0]
    df_spam = df[df.label==1]

    bigger = max(len(df_ham), len(df_spam))

    df_ham  = resample(df_ham,  replace=True, n_samples=bigger, random_state=42)
    df_spam = resample(df_spam, replace=True, n_samples=bigger, random_state=42)

    df = pd.concat([df_ham, df_spam])
    df = df.sample(frac=1).reset_index(drop=True)

    print("Balanced distribution:")
    print(df.label.value_counts())

In [13]:
# =========================================
# 11. LIGHT DATA NOISE (ANTI-OVERFITTING)
# =========================================
if not model_exists:

    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)

    def add_noise(text):
        words = text.split()
        if len(words) > 3 and random.random() < 0.12:
            idx = random.randint(0, len(words)-1)
            words[idx] = "noise"
        return " ".join(words)

    def flip_label(label):
        if random.random() < 0.03:
            return 1 - label
        return label

    df["SMS"]   = df["SMS"].apply(add_noise)
    df["label"] = df["label"].apply(flip_label)

In [14]:
# =========================================
# 12. TRAIN TEST SPLIT
# =========================================
if not model_exists:

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        df["SMS"].tolist(),
        df["label"].tolist(),
        test_size=0.2,
        random_state=42
    )



In [15]:
# =========================================
# 13. TOKENIZATION
# =========================================
if not model_exists:

    tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

    train_encodings = tokenizer(
        train_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

    val_encodings = tokenizer(
        val_texts,
        truncation=True,
        padding=True,
        max_length=128
    )

In [16]:
# =========================================
# 14. DATASET CLASS
# =========================================
if not model_exists:

    class SMSDataset(Dataset):

        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels

        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item

        def __len__(self):
            return len(self.labels)

    train_dataset = SMSDataset(train_encodings, train_labels)
    val_dataset   = SMSDataset(val_encodings,   val_labels)

In [17]:
# =========================================
# 15. TRAIN BERT
# =========================================
if not model_exists:

    model = BertForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    )

    model.to(device)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=16)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

    epochs = 3

    num_training_steps = len(train_loader) * epochs

    scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps
    )

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for batch in train_loader:

            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            total_loss += loss.item()

            loss.backward()

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

In [18]:
# =========================================
# 16. SAVE MODEL TO DRIVE
# =========================================
if not model_exists:

    model.save_pretrained(MODEL_PATH)
    tokenizer.save_pretrained(MODEL_PATH)

    model_exists = True  # update flag so next cells know model is ready

    print("Model saved to Google Drive!")

In [19]:
# =========================================
# 17. EVALUATE MODEL
# =========================================

# rebuild val_loader if model was loaded from Drive
if not 'val_loader' in dir():

    url7 = "https://raw.githubusercontent.com/War-rack/Q-Classify/main/spam.csv"
    df_eval = pd.read_csv(url7, encoding="latin1")
    df_eval = df_eval.rename(columns={"v1":"label","v2":"SMS"})[["label","SMS"]]
    df_eval.dropna(inplace=True)
    df_eval.drop_duplicates(inplace=True)
    df_eval["label"] = df_eval["label"].replace({"ham":0,"spam":1}).astype(int)

    _, val_texts_e, _, val_labels_e = train_test_split(
        df_eval["SMS"].tolist(), df_eval["label"].tolist(),
        test_size=0.2, random_state=42
    )

    val_enc = tokenizer(val_texts_e, truncation=True, padding=True, max_length=128)

    class SMSDataset(Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
        def __getitem__(self, idx):
            item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx])
            return item
        def __len__(self):
            return len(self.labels)

    val_loader = DataLoader(SMSDataset(val_enc, val_labels_e), batch_size=16)
    print(f"Val loader ready — {len(val_labels_e)} samples")

model.eval()
preds, true = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=1)
        preds.extend(predictions.cpu().numpy())
        true.extend(batch["labels"].cpu().numpy())

print("Validation Accuracy:", accuracy_score(true, preds))
print("\nClassification Report:\n")
print(classification_report(true, preds, target_names=["HAM","SPAM"]))

/tmp/ipykernel_816/1487081294.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_eval["label"] = df_eval["label"].replace({"ham":0,"spam":1}).astype(int)


Val loader ready — 1034 samples
Validation Accuracy: 0.9932301740812379

Classification Report:

              precision    recall  f1-score   support

         HAM       1.00      0.99      1.00       889
        SPAM       0.96      0.99      0.98       145

    accuracy                           0.99      1034
   macro avg       0.98      0.99      0.99      1034
weighted avg       0.99      0.99      0.99      1034



In [20]:
# =========================================
# UNSEEN DATA TEST EVALUATION
# =========================================

# Load a fresh dataset - using Telegram spam dataset (url2 from your training)
# We'll use a portion that was NOT part of training split
df_unseen = pd.read_csv(
    "https://huggingface.co/datasets/thehamkercat/telegram-spam-ham/resolve/main/dataset.csv"
)
df_unseen = df_unseen.rename(columns={"text_type":"label","text":"SMS"})[["label","SMS"]]
df_unseen.dropna(inplace=True)
df_unseen.drop_duplicates(inplace=True)
df_unseen["label"] = df_unseen["label"].replace({"ham":0,"spam":1}).astype(int)

print(f"Unseen dataset size: {df_unseen.shape}")
print(df_unseen["label"].value_counts())

# Tokenize full unseen dataset
test_texts = df_unseen["SMS"].tolist()
test_labels = df_unseen["label"].tolist()

test_enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=128
)

class SMSDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

test_loader = DataLoader(SMSDataset(test_enc, test_labels), batch_size=16)

# Run evaluation
model.eval()
preds, true = [], []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        predictions = torch.argmax(outputs.logits, dim=1)
        preds.extend(predictions.cpu().numpy())
        true.extend(batch["labels"].cpu().numpy())

# Results
print("\n✅ Test Accuracy:", accuracy_score(true, preds))
print("\n📊 Classification Report:\n")
print(classification_report(true, preds, target_names=["HAM","SPAM"]))

/tmp/ipykernel_816/585258078.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_unseen["label"] = df_unseen["label"].replace({"ham":0,"spam":1}).astype(int)


Unseen dataset size: (20334, 2)
label
0    14323
1     6011
Name: count, dtype: int64

✅ Test Accuracy: 0.9899183633323497

📊 Classification Report:

              precision    recall  f1-score   support

         HAM       1.00      0.99      0.99     14323
        SPAM       0.98      0.99      0.98      6011

    accuracy                           0.99     20334
   macro avg       0.99      0.99      0.99     20334
weighted avg       0.99      0.99      0.99     20334



In [21]:
# =========================================
# 18. PREDICTION WITH RISK SCORE
# =========================================

def predict_message(message):

    inputs = tokenizer(
        message,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs   = torch.softmax(outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1).item()

    spam_prob  = float(probs[0][1])
    risk_score = round(spam_prob * 100, 2)
    label      = "SPAM" if pred == 1 else "HAM"

    return {
        "message": message,
        "prediction": label,
        "spam_probability": spam_prob,
        "risk_score": risk_score
    }

In [22]:
# =========================================
# 19. TEST PREDICTIONS
# =========================================

msgs = [
    """Hi Charu,

You started an application for Intern - Technical – 498774, but it has not yet been submitted.

Your application is currently saved as a draft. Please note that draft applications are kept for up to 30 days from the date they are first saved OR while the position remains open. Only submitted applications can be considered for this role.

You can continue and submit your application at any time by visiting your Dashboard here and completing the remaining steps.

We look forward to receiving your application.

Best regards,
Siemens Talent Team""",

 "Schedule Company Name - Josh Technology Date of Online Test - 3rd Dec 2025",

]

for msg in msgs:
    result = predict_message(msg)
    print(f"[{result['prediction']}] Risk: {result['risk_score']}% | {msg[:60]}...")
    print()

[HAM] Risk: 3.36% | Hi Charu,

You started an application for Intern - Technical...

[HAM] Risk: 3.52% | Schedule Company Name - Josh Technology Date of Online Test ...



In [23]:
msg = "Schedule Company Name - Josh Technology Date of Online Test - 3rd Dec 2025 Reporting Time - 3:30 PM ( Late entries will not be allowed to participate )Venue - E2 Block, 5th Floor Labs. Important Note -It is mandatory for all students to appear for the Online Test from the designated campus labs and strictly as per the slot assigned to them.Any student found involved in malpractice or attempting the Online Test from a remote location will be debarred from all further placement drives.All the Best !!Dr. Anjani Kumar Bhatnagar"

print(predict_message(msg))


{'message': 'Schedule Company Name - Josh Technology Date of Online Test - 3rd Dec 2025 Reporting Time - 3:30 PM ( Late entries will not be allowed to participate )Venue - E2 Block, 5th Floor Labs. Important Note -It is mandatory for all students to appear for the Online Test from the designated campus labs and strictly as per the slot assigned to them.Any student found involved in malpractice or attempting the Online Test from a remote location will be debarred from all further placement drives.All the Best !!Dr. Anjani Kumar Bhatnagar', 'prediction': 'HAM', 'spam_probability': 0.09918589144945145, 'risk_score': 9.92}


In [24]:
msg="""
Dr. Anjani K Bhatnagar <ajbhatnagar@amity.edu>
Sat, Mar 28, 9:55 AM
to ATPC
Dear Students
In continuation of your registration on Amizone for the Mahindra Automotive Virtual Recruitment Drive. This is to inform you that the registration process for the Mahindra Campus Hiring drive has been initiated.
Do not share the below-mentioned registration link with anyone in the group. This is exclusively shared with registered and eligible students.
You are required to complete the registration using the link below:
https://mycareernet.co/events/mahindra-campus-hiring-2026/
Registration window is open and closes on 5th April 2026, 11 pm.
Post registration, applications will be reviewed as per the defined eligibility criteria, and shortlisted candidates will be informed regarding the subsequent stages of the selection process.
My Best Wishes are with you
Dr. Anjani Kumar Bhatnagar
PhD (Management), MBA & PGDHRM"""

print(predict_message(msg))



{'message': '\nDr. Anjani K Bhatnagar <ajbhatnagar@amity.edu>\nSat, Mar 28, 9:55\u202fAM\nto ATPC\nDear Students\nIn continuation of your registration on Amizone for the Mahindra Automotive Virtual Recruitment Drive. This is to inform you that the registration process for the Mahindra Campus Hiring drive has been initiated.\nDo not share the below-mentioned registration link with anyone in the group. This is exclusively shared with registered and eligible students.\nYou are required to complete the registration using the link below:\nhttps://mycareernet.co/events/mahindra-campus-hiring-2026/\nRegistration window is open and closes on 5th April 2026, 11 pm.\nPost registration, applications will be reviewed as per the defined eligibility criteria, and shortlisted candidates will be informed regarding the subsequent stages of the selection process.\nMy Best Wishes are with you\nDr. Anjani Kumar Bhatnagar\nPhD (Management), MBA & PGDHRM', 'prediction': 'HAM', 'spam_probability': 0.26235529

In [25]:
msg="""From: People’s Bank [service@peoples.com]
Subject: Update Notifi cation
 Dear People’s Bank® Customer,
Recently there have been a large number of identity theft attempts targeting our customers. In order to safeguard
your account, we require that you confi rm your banking details.
This process is mandatory, and if not completed within the nearest time your account or credit card may be
subject to temporary suspension.
To securely confi rm your People’s Bank Account details please follow the link:
LINK REMOVED FROM THIS EXAMPLE.
Note: You may have to report this message as “Not Junk Mail” if update link does not work.
Thank you for your prompt attention to this matter and thank you for using People’s Bank.
© Copyright 2005, People’s Bank, Inc. All Rights Reserved. """

print(predict_message(msg))



{'message': 'From: People’s Bank [service@peoples.com]\nSubject: Update Notifi cation\n Dear People’s Bank® Customer,\nRecently there have been a large number of identity theft attempts targeting our customers. In order to safeguard\nyour account, we require that you confi rm your banking details.\nThis process is mandatory, and if not completed within the nearest time your account or credit card may be\nsubject to temporary suspension.\nTo securely confi rm your People’s Bank Account details please follow the link:\nLINK REMOVED FROM THIS EXAMPLE.\nNote: You may have to report this message as “Not Junk Mail” if update link does not work.\nThank you for your prompt attention to this matter and thank you for using People’s Bank.\n© Copyright 2005, People’s Bank, Inc. All Rights Reserved. ', 'prediction': 'SPAM', 'spam_probability': 0.9780774116516113, 'risk_score': 97.81}


In [26]:
msg="""From: Peoples Bank
Subject: EFT Activity Notifi cation(Your payment is processing)
Your payment($495.42) is processing .
To verify that your payment was applied follow the steps below:
1. Access URL REMOVED FROM THIS EXAMPLE FOR SECURITY REASONS.
2. Login and select Charter e-pay
3. Select Recent Activity on the left navigation
4. View Payment Status
Your one time EFT debit card payment dated Sunday 06/07/2005 MST is listed below.
Expiration Date: Sunday 06/07/2005
Payment: $495.42
PLEASE NOTE: THIS PAYMENT IS A ‘REAL TIME’ PAYMENT PAID DIRECTLY FROM YOUR CHECKING
ACCOUNT WHEN YOU SUBMITTED YOUR PAYMENT.
Thank you again for making a ONE TIME payment online.
This email message is not set up to receive reply messages. If you have any questions regarding this email
notifi cation of your account change, please contact us at URL REMOVED FROM THIS EXAMPLE FOR
SECURITY REASONS.
Thank you for using Peoples Bank Online for this request.
Peoples Bank
 """

print(predict_message(msg))



{'message': 'From: Peoples Bank\nSubject: EFT Activity Notifi cation(Your payment is processing)\nYour payment($495.42) is processing .\nTo verify that your payment was applied follow the steps below:\n1. Access URL REMOVED FROM THIS EXAMPLE FOR SECURITY REASONS.\n2. Login and select Charter e-pay\n3. Select Recent Activity on the left navigation\n4. View Payment Status\nYour one time EFT debit card payment dated Sunday 06/07/2005 MST is listed below.\nExpiration Date: Sunday 06/07/2005\nPayment: $495.42\nPLEASE NOTE: THIS PAYMENT IS A ‘REAL TIME’ PAYMENT PAID DIRECTLY FROM YOUR CHECKING\nACCOUNT WHEN YOU SUBMITTED YOUR PAYMENT.\nThank you again for making a ONE TIME payment online.\nThis email message is not set up to receive reply messages. If you have any questions regarding this email\nnotifi cation of your account change, please contact us at URL REMOVED FROM THIS EXAMPLE FOR\nSECURITY REASONS.\nThank you for using Peoples Bank Online for this request.\nPeoples Bank\n ', 'predicti